# Lab 11/12 — The three formats, measured

Save **one** model three ways and measure what each flattening costs and discards.
Read the [lab brief](Lab11_12.md) first. Marked on **relationships between your own
numbers**, not their magnitudes — your disk and allocator are yours.

**Before you run anything:** set your identity in the next cell. Fill every 📝 cell.
Run top to bottom, then run the final export cell and submit the two files it names.

In [1]:
ROLL_NUMBER = "202518030"
NAME        = "Dhruv Parmar"

# Real .gguf for Part 4: the Ollama blob for qwen2.5:7b (Q4_K_M), mounted read-only into the
# Linux container at this path. On the host it is ~/.ollama/models/blobs/sha256-2bada8a7...
GGUF_PATH   = "/gguf/qwen2.5-7b-instruct-q4_k_m.gguf"

assert ROLL_NUMBER and NAME, "Set ROLL_NUMBER and NAME before running the rest."


In [2]:
# Colab: uncomment.
# !pip -q install torch safetensors

import os, sys, time, json, struct, platform, pickle, subprocess, textwrap
from pathlib import Path
import torch, torch.nn as nn
from safetensors.torch import save_file, load_file
from safetensors import safe_open
import safetensors

WORK = Path("lab11_12_work"); WORK.mkdir(exist_ok=True)
RESULTS = {"formats": {}, "stride": {}, "trust": {}, "gguf_real": {}}
IS_LINUX = sys.platform.startswith("linux")

def rss_mb():
    "Current resident memory (VmRSS). ru_maxrss is a peak and never falls, so it is useless here."
    if IS_LINUX:
        for l in open("/proc/self/status"):
            if l.startswith("VmRSS:"): return int(l.split()[1]) / 1024
    import resource; return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

def io_field(name):
    "A counter from /proc/self/io. rchar = bytes moved by read() syscalls; read_bytes = bytes off the block device."
    if not IS_LINUX: return None
    for l in open("/proc/self/io"):
        if l.startswith(name): return int(l.split()[1])
    return None

def evict(path):
    "Drop this file's pages from the page cache — rootless, Linux only. Turns the next read cold."
    if not IS_LINUX: return
    fd = os.open(path, os.O_RDONLY)
    # FIX: flush writeback first. Dirty/under-writeback pages are not dropped by DONTNEED, so right
    # after torch.save the 'cold' load could be served warm (measured: 56 of 240 MB dropped).
    try: os.fsync(fd); os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
    finally: os.close(fd)

RSS_PROBE = r"""
import sys, gc
def rss(): return int(next(l for l in open("/proc/self/status") if l.startswith("VmRSS")).split()[1]) / 1024
fmt, path = sys.argv[1], sys.argv[2]
import torch
from safetensors.torch import load_file
import gguf
gc.collect(); r0 = rss()                      # imports are paid BEFORE the baseline
if fmt == "pt":            obj = torch.load(path, weights_only=True)
elif fmt == "safetensors": obj = load_file(path)
else:
    rd = gguf.GGUFReader(path); obj = {t.name: t.data for t in rd.tensors}
print(rss() - r0)
"""

def fresh_process_rss_delta(fmt, path):
    """FIX: resident-memory delta of ONE load, measured in a brand-new interpreter.

    PyTorch's CPU allocator keeps freed tensor memory resident and reuses it (measured here:
    torch.empty(251 MB) freed -> still +241.7 MB resident, while glibc malloc and numpy return it).
    The in-process measurement is taken on the 3rd load, after two freed loads, so it reuses that
    retained memory and reads ~0 for .pt. A fresh process has nothing to reuse."""
    out = subprocess.run([sys.executable, "-c", RSS_PROBE, fmt, str(path)],
                         capture_output=True, text=True, check=True)
    return float(out.stdout.split()[-1])

print("torch", torch.__version__, "| safetensors", safetensors.__version__, "| linux:", IS_LINUX)


torch 2.14.0+cpu | safetensors 0.8.0 | linux: True


### Note: three changes I made to the provided measurement helpers, and why

My first run (in Linux, so the `/proc` counters are real) gave `.pt` **RSS +0.0 MB**, a **cold `.pt` load that read 0 MB from disk**, and **mmap load times of 0.000 s**. Before explaining those numbers I checked whether they measured what they claim, using small probe scripts (`scripts/diagnose_*.py`, outputs in `README.md`):

1. **Eviction was racing writeback.** Right after `torch.save` about 241 MB is still dirty or under writeback, and `POSIX_FADV_DONTNEED` cannot drop those pages. Two identical runs dropped **56 MB** and **240 MB** of the same 252 MB `.pt`. → `evict()` now calls `fsync` first, so every cold load is really cold.
2. **PyTorch's CPU allocator keeps freed memory.** `torch.empty(251 MB)` freed still leaves **+241.7 MB** resident (glibc `malloc` and NumPy return it). The provided code measures RSS on the **third** load, after two freed loads, so it reuses retained memory and reads ~0 even though a first load adds **~226–244 MB**. → `load_rss_delta_mb` is now measured on one load in a **fresh interpreter** for every format. The original in-process number is kept as `load_rss_delta_inprocess_mb`.
3. **mmap loads do not read data.** `load_file` / `GGUFReader` return in ~0 s whatever the cache state, so their cold/warm times cannot show the page cache. → I added `cold_touch_s` / `warm_touch_s`, which time a load that reads every weight.

The model, file writes, loads and grading fields are otherwise unchanged.


In [3]:
# One model, ~200 MB of fp32 weights. Big enough that cache and copy effects are visible.
torch.manual_seed(635)
model = nn.Sequential(nn.Linear(8192, 6144), nn.Linear(6144, 2048))
sd = model.state_dict()
total_mb = sum(t.numel()*t.element_size() for t in sd.values()) / 1e6
print(f"model: {sum(t.numel() for t in sd.values())/1e6:.1f}M params, {total_mb:.0f} MB of weights")


model: 62.9M params, 252 MB of weights


---
## Part 1 — The load path

📝 **Predict, before running the next cell.** For `.pt` and `.safetensors`, which loads
faster warm than cold? Which leaves resident memory roughly equal to the file size, and
which barely moves it? Which shows bytes read from disk on a cold load? Write your
predictions and one sentence of reasoning each.

**My predictions (written before running):**

- **Warm vs cold:** both formats will load faster warm than cold. `evict()` drops the file's pages from the page cache, so the cold load has to fetch ~200 MB from the disk. The warm load finds the pages already in RAM and only copies or maps them.
- **Resident memory:** the `.pt` load will raise RSS by about the file size (~200 MB). Unpickling rebuilds the whole object graph, and `torch.load` copies every storage out of the zip into freshly allocated tensor memory. The `.safetensors` load will barely move RSS (a few MB at most), because `load_file` `mmap`s the file and the tensors are views over mapped pages that are not touched yet.
- **Bytes read on a cold load:** the `.pt` cold load will show `read_bytes` ≈ file size, since every storage is read. For `.safetensors`, `read_bytes` should be much smaller than the file (roughly the header plus whatever pages get faulted), because mmap reads pages only on demand. `rchar` will be ≈ file size for `.pt` (it goes through `read()` calls on the zip) and ≈ 0 for the safetensors load. The explicit `path.read_bytes()` of the same safetensors file will show `rchar` ≈ file size.


In [4]:
def measure_pt(sd, path):
    torch.save(sd, path)
    fmb = path.stat().st_size/1e6
    evict(path); rb0=io_field("read_bytes"); t=time.perf_counter(); torch.load(path, weights_only=True); cold=time.perf_counter()-t; cold_rb=(io_field("read_bytes")-rb0)/1e6 if rb0 is not None else None
    t=time.perf_counter(); torch.load(path, weights_only=True); warm=time.perf_counter()-t
    r0=rss_mb(); rc0=io_field("rchar"); obj=torch.load(path, weights_only=True); rss=rss_mb()-r0; rc=(io_field("rchar")-rc0)/1e6 if rc0 is not None else None
    del obj
    return dict(file_mb=round(fmb,1), cold_s=round(cold,3), warm_s=round(warm,3),
                cold_read_bytes_mb=round(cold_rb,1) if cold_rb is not None else None,
                load_rss_delta_mb=round(fresh_process_rss_delta('pt', path) if IS_LINUX else rss, 1),
                load_rss_delta_inprocess_mb=round(rss,1), load_rchar_mb=round(rc,1) if rc is not None else None)

def measure_safetensors(sd, path):
    save_file(sd, str(path))
    fmb = path.stat().st_size/1e6
    evict(path); rb0=io_field("read_bytes"); t=time.perf_counter(); load_file(path); cold=time.perf_counter()-t; cold_rb=(io_field("read_bytes")-rb0)/1e6 if rb0 is not None else None
    t=time.perf_counter(); load_file(path); warm=time.perf_counter()-t
    r0=rss_mb(); rc0=io_field("rchar"); obj=load_file(path); rss=rss_mb()-r0; rc=(io_field("rchar")-rc0)/1e6 if rc0 is not None else None
    del obj
    # explicit read() of the same bytes, for the rchar contrast
    rc0=io_field("rchar"); _=path.read_bytes(); read_rc=(io_field("rchar")-rc0)/1e6 if rc0 is not None else None
    # FIX: load_file only maps the file, so its cold/warm times are ~0 whatever the cache holds.
    # Also time a load that reads every page, so the page cache is visible.
    def touch(): return sum(float(t.sum()) for t in load_file(path).values())
    evict(path); t=time.perf_counter(); touch(); cold_touch=time.perf_counter()-t
    t=time.perf_counter(); touch(); warm_touch=time.perf_counter()-t
    return dict(file_mb=round(fmb,1), cold_s=round(cold,3), warm_s=round(warm,3),
                cold_read_bytes_mb=round(cold_rb,1) if cold_rb is not None else None,
                load_rss_delta_mb=round(fresh_process_rss_delta('safetensors', path) if IS_LINUX else rss, 1),
                load_rss_delta_inprocess_mb=round(rss,1),
                cold_touch_s=round(cold_touch,3), warm_touch_s=round(warm_touch,3), load_rchar_mb=round(rc,1) if rc is not None else None,
                explicit_read_rchar_mb=round(read_rc,1) if read_rc is not None else None)

RESULTS["formats"]["pt"] = measure_pt(sd, WORK/"model.pt")
RESULTS["formats"]["safetensors"] = measure_safetensors(sd, WORK/"model.safetensors")
if GGUF_PATH and Path(GGUF_PATH).exists():
    RESULTS["formats"]["gguf"] = dict(file_mb=round(Path(GGUF_PATH).stat().st_size/1e6,1))

for k, v in RESULTS["formats"].items():
    print(f"{k:<12}", v)


pt           {'file_mb': 251.7, 'cold_s': 0.021, 'warm_s': 0.011, 'cold_read_bytes_mb': 251.7, 'load_rss_delta_mb': 246.8, 'load_rss_delta_inprocess_mb': 0.0, 'load_rchar_mb': 251.7}
safetensors  {'file_mb': 251.7, 'cold_s': 0.0, 'warm_s': 0.0, 'cold_read_bytes_mb': 0.4, 'load_rss_delta_mb': 2.0, 'load_rss_delta_inprocess_mb': 0.1, 'cold_touch_s': 0.028, 'warm_touch_s': 0.003, 'load_rchar_mb': 0.0, 'explicit_read_rchar_mb': 251.7}
gguf         {'file_mb': 4683.1}


In [5]:
# Part 1, third format: the SAME state_dict written as .gguf with gguf-py (llama.cpp's writer),
# then loaded through gguf.GGUFReader, which np.memmap()s the file exactly as llama.cpp mmaps it.
import gguf, numpy as np, gc

def measure_gguf(sd, path):
    w = gguf.GGUFWriter(str(path), arch="ds635_mlp")
    w.add_name("ds635-lab11-12-mlp"); w.add_block_count(len(model))
    for name, t in sd.items():
        w.add_tensor(name, t.detach().contiguous().numpy())          # F32, row-major, no strides stored
    w.write_header_to_file(); w.write_kv_data_to_file(); w.write_tensors_to_file(); w.close()
    fmb = path.stat().st_size/1e6

    def load():
        r = gguf.GGUFReader(str(path))
        return {t.name: t.data for t in r.tensors}                    # np.memmap views, no copy

    evict(path); rb0=io_field("read_bytes"); t=time.perf_counter(); o=load(); cold=time.perf_counter()-t; cold_rb=(io_field("read_bytes")-rb0)/1e6 if rb0 is not None else None
    del o; gc.collect()
    t=time.perf_counter(); o=load(); warm=time.perf_counter()-t; del o; gc.collect()
    r0=rss_mb(); rc0=io_field("rchar"); o=load(); rss=rss_mb()-r0; rc=(io_field("rchar")-rc0)/1e6 if rc0 is not None else None
    # sanity: the memmapped bytes are the model's bytes
    def touch(): return sum(float(np.asarray(a, dtype=np.float64).sum()) for a in load().values())
    evict(path); t=time.perf_counter(); touch(); cold_touch=time.perf_counter()-t
    t=time.perf_counter(); touch(); warm_touch=time.perf_counter()-t
    same = all(np.array_equal(np.asarray(o[k]).reshape(v.shape), v.numpy()) for k, v in sd.items())
    del o; gc.collect()
    return dict(file_mb=round(fmb,1), cold_s=round(cold,3), warm_s=round(warm,3),
                cold_read_bytes_mb=round(cold_rb,1) if cold_rb is not None else None,
                load_rss_delta_mb=round(fresh_process_rss_delta('gguf', path) if IS_LINUX else rss, 1),
                load_rss_delta_inprocess_mb=round(rss,1),
                cold_touch_s=round(cold_touch,3), warm_touch_s=round(warm_touch,3), load_rchar_mb=round(rc,1) if rc is not None else None,
                bytes_identical_to_state_dict=same)

RESULTS["formats"]["gguf"] = measure_gguf(sd, WORK/"model.gguf")
if GGUF_PATH and Path(GGUF_PATH).exists():
    RESULTS["gguf_real"]["file_mb"] = round(Path(GGUF_PATH).stat().st_size/1e6, 1)

hdr = ("format", "file MB", "cold s", "warm s", "cold+touch s", "warm+touch s",
       "cold rd MB", "RSS+ fresh", "RSS+ in-proc", "rchar MB")
print("".join(f"{h:>13}" for h in hdr))
for k, v in RESULTS["formats"].items():
    row = (k, v["file_mb"], v["cold_s"], v["warm_s"], v.get("cold_touch_s", "-"), v.get("warm_touch_s", "-"),
           v.get("cold_read_bytes_mb"), v["load_rss_delta_mb"], v.get("load_rss_delta_inprocess_mb"), v.get("load_rchar_mb"))
    print("".join(f"{str(x):>13}" for x in row))
print("safetensors explicit read() rchar MB:", RESULTS["formats"]["safetensors"]["explicit_read_rchar_mb"])


       format      file MB       cold s       warm s cold+touch s warm+touch s   cold rd MB   RSS+ fresh RSS+ in-proc     rchar MB
           pt        251.7        0.021        0.011            -            -        251.7        246.8          0.0        251.7
  safetensors        251.7          0.0          0.0        0.028        0.003          0.4          2.0          0.1          0.0
         gguf        251.7          0.0          0.0        0.066        0.023          0.1          0.3          0.1          0.0
safetensors explicit read() rchar MB: 251.7


📝 **Explain your numbers.** Warm vs cold: what did the second load skip? `.pt` vs
`.safetensors` resident memory: why does one match the file size and the other not?
And why did `.safetensors` load with `rchar ≈ 0` while an explicit `read()` of the same
file moved the whole file? Name the mechanism in each case.

**What I measured** (Linux container, after the three fixes in the note at the top):

| | `.pt` | `.safetensors` | `.gguf` (same weights) |
|---|---|---|---|
| file size | 251.7 MB | 251.7 MB | 251.7 MB |
| cold / warm load | 0.021 / 0.011 s | 0.0 / 0.0 s | 0.0 / 0.0 s |
| cold / warm load **+ read every weight** | (load already reads everything) | 0.028 / 0.003 s | 0.066 / 0.023 s |
| `read_bytes` on the cold load | 251.7 MB | 0.4 MB | 0.1 MB |
| RSS delta, one load in a fresh process | **+246.8 MB** | **+2.0 MB** | **+0.3 MB** |
| RSS delta, provided in-process method (3rd load) | +0.0 MB | +0.1 MB | +0.1 MB |
| `rchar` during the load | 251.7 MB | **0.0 MB** | 0.0 MB |
| `rchar` for `path.read_bytes()` of the same file | | **251.7 MB** | |

(The `gguf 4683.1 MB` printed by the provided cell is the size of the real Qwen2.5 file at `GGUF_PATH`, a different model. The next cell replaces it with this model written as `.gguf`.)

**Warm vs cold. Mechanism: the page cache.** After `fsync` + `POSIX_FADV_DONTNEED`, the cold `.pt` load had to pull all 251.7 MB off the block device (`read_bytes`). The warm load found the same pages already in RAM, so `read()` became a memory copy: 0.021 s → 0.011 s. For safetensors the bare `load_file` took ~0 s cold *and* warm, because an mmap load reads nothing up front, so there is nothing for the cache to speed up. Once every weight is actually read, the difference appears: 0.028 s cold vs 0.003 s warm. Cold page faults turn into disk reads; warm faults just map pages that are already cached. The `.pt` gap is only ~2× because this "disk" is a VM image on the Mac's NVMe, which delivers 252 MB in about 10 ms, and much of even the cold `.pt` load is the allocator copying into ~250 MB of fresh anonymous memory. That copy costs the same warm or cold.

**Resident memory. Mechanisms: allocator copy vs mmap.** `.pt` grew RSS by 246.8 MB ≈ the file size. Unpickling rebuilds the object graph, and `torch.load` reads each storage record out of the zip and copies it into newly allocated, private tensor memory, so the whole model now exists a second time in anonymous pages. `.safetensors` grew RSS by only 2.0 MB (and GGUF by 0.3 MB). `load_file` `mmap`s the file, and each tensor is a view at its `data_offsets` into file-backed pages that nobody has touched yet. RSS only grows as pages are faulted in, and those are page-cache pages the kernel can drop and re-read, not private copies. The provided in-process measurement showed `.pt` at +0.0 MB. That is not the format being cheap: PyTorch's CPU allocator kept the ~250 MB freed by the two earlier loads and handed it to the third (probed in `diagnostics_log.md`: a freed `torch.empty(251 MB)` stays resident, while glibc and NumPy return theirs). This is why I measured one load in a fresh process.

**`rchar ≈ 0` for safetensors vs 251.7 MB for `read()`. Mechanism: mmap issues no read syscall.** `rchar` counts bytes passed through `read()`-family syscalls. `.pt` loading goes through `read()` on the zip (251.7 MB). `path.read_bytes()` is one big `read()` of the whole file. An mmap load maps the file into the address space and bytes arrive later through page faults, which count toward `read_bytes` when they hit the disk but never toward `rchar`. The 0.4 MB of `read_bytes` on the cold safetensors load is the only part the loader actually looked at: the 8-byte length, the JSON header and the readahead around it.


---
## Part 2 — The stride tax

📝 **Predict.** Transposing a big tensor — how many bytes move? Calling `.contiguous()`
on that transpose — how should its cost scale with tensor size? Will `safetensors` save a
transposed view?

**My predictions (written before running):**

- **Transpose:** zero bytes move. `big.T` only swaps the stride tuple from `(8192, 1)` to `(1, 8192)` and returns a view on the same storage, so `data_ptr()` will be identical.
- **`.contiguous()`:** this allocates a new buffer and physically copies every element into row-major order. Its cost should grow roughly linearly with the number of bytes (≈ n²·4). Going 2048² → 8192² is 16× the bytes, so I expect roughly 10–20× the time. It may be a bit superlinear once the tensor no longer fits in cache, because the transposed read order walks down columns.
- **safetensors and the view:** it will refuse `big.T`. The format stores only `dtype`, `shape` and `data_offsets`, never strides, so it can only describe tightly packed row-major bytes. The packed `.contiguous()` copy will be accepted.


In [6]:
big = torch.randn(8192, 8192)
v = big.T
RESULTS["stride"]["transpose_same_ptr"] = bool(v.data_ptr() == big.data_ptr())

sizes, times = [], []
for n in (2048, 4096, 6144, 8192):
    a = torch.randn(n, n)
    t = time.perf_counter(); _ = a.T.contiguous(); dt = (time.perf_counter()-t)*1e3
    sizes.append(round(n*n*4/1e6, 1)); times.append(round(dt, 2))
RESULTS["stride"]["contig_sizes_mb"] = sizes
RESULTS["stride"]["contig_times_ms"] = times

try:
    save_file({"w": big.T}, str(WORK/"probe.safetensors")); refused = False
except Exception: refused = True
save_file({"w": big.contiguous()}, str(WORK/"probe.safetensors"))   # the packed copy is accepted
RESULTS["stride"]["safetensors_refused_view"] = refused

print("transpose shares buffer:", RESULTS["stride"]["transpose_same_ptr"])
print("contiguous cost (MB -> ms):", list(zip(sizes, times)))
print("safetensors refused the view:", refused)


transpose shares buffer: True
contiguous cost (MB -> ms): [(16.8, 2.23), (67.1, 11.52), (151.0, 30.82), (268.4, 47.85)]
safetensors refused the view: True


📝 **Explain.** Who paid for contiguity — the machine saving the model, or the machine
loading it? Why is that the right place to put the cost for a file loaded far more often
than it is written?

**Measured:** `big.T` kept the same `data_ptr` (`transpose_same_ptr = True`). Zero bytes moved; only the strides changed from `(8192, 1)` to `(1, 8192)`. `.contiguous()` on the transpose cost 2.23 ms for 16.8 MB, 11.52 ms for 67.1 MB, 30.82 ms for 151.0 MB, 47.85 ms for 268.4 MB. That is 21.5× the time for 16× the bytes (0.13 → 0.18 ms/MB), so it grows with the bytes moved and is slightly worse than linear. The source is read down columns (stride 8192), which touches a new cache line for almost every element: memory traffic, not arithmetic. safetensors **refused** the view (`True`) and accepted the packed copy.

**Who paid: the machine saving the model.** A safetensors header holds only `dtype`, `shape` and `data_offsets`, with no strides. So the file can only describe tightly packed row-major bytes, and the writer had to run `.contiguous()` first: ~48 ms and a second 268 MB buffer here. The reader never pays. It maps the bytes and derives the strides from the shape (`[8192, 1]` for this 2-D tensor), with no copy and no layout decision.

**Why that is the right place.** A model is written once, usually on a machine with plenty of RAM, and then loaded thousands to millions of times on laptops, phones and servers. Packing on save is a one-time, bounded cost. Supporting arbitrary strides in the file would push work onto every load: each reader would have to understand views and often pack them itself, which defeats zero-copy mmap loading. Canonical contiguous bytes are also what hardware reads fastest, as one sequential stride-1 sweep (Lecture 8's coalescing). Refusing the view, instead of silently copying, makes the cost visible to the one party who can pay it once: the writer.


---
## Part 3 — The trust boundary

📝 **Predict.** A pickle whose `__reduce__` runs code — will `torch.load` fire it under the
2.6 default? Under `weights_only=False`?

**My predictions (written before running):**

- **`weights_only=True` (the default since PyTorch 2.6):** it will **not** fire. The weights-only unpickler only resolves globals on its allow-list (tensor rebuild functions, basic containers). `os.system` is not on the list, so `torch.load` raises an `UnpicklingError` before calling anything, and `PWNED.txt` will not exist.
- **`weights_only=False`:** it **will** fire. The full pickle machinery resolves `os.system` by name (`GLOBAL`/`STACK_GLOBAL`) and calls it (`REDUCE`) while deserialising. The load itself is the execution, so `PWNED.txt` will appear even though my code never touches the `"x"` key.


In [7]:
PWNED = WORK/"PWNED.txt"

class Benign:
    "Harmless: __reduce__ names a callable; loading calls it. Swap os.system for the real thing."
    def __reduce__(self):
        return (os.system, (f'echo "payload ran" > {PWNED}',))

torch.save({"w": torch.zeros(2), "x": Benign()}, WORK/"evil.pt")

PWNED.unlink(missing_ok=True)
try:
    torch.load(WORK/"evil.pt", weights_only=True); fired_default = PWNED.exists(); blocked = False
except Exception:
    fired_default = PWNED.exists(); blocked = True

PWNED.unlink(missing_ok=True)
torch.load(WORK/"evil.pt", weights_only=False); fired_unsafe = PWNED.exists()

RESULTS["trust"] = dict(payload_fired_default=bool(fired_default),
                        blocked_by_weights_only=bool(blocked),
                        payload_fired_unsafe=bool(fired_unsafe))
print(RESULTS["trust"])


{'payload_fired_default': False, 'blocked_by_weights_only': True, 'payload_fired_unsafe': True}


📝 **The paragraph that carries this part.** `safetensors` removes the execution
mechanism. Name **at least two** things a `.safetensors` checkpoint still cannot protect
you from, and say why the format has no opinion on them.

**My answer:**

Safetensors fixes exactly one thing: its file is a u64 header length, a JSON header of `{dtype, shape, data_offsets}`, and raw bytes. There are no opcodes, no callable names and no type registry, so *loading* cannot run code by construction. That is stronger than `weights_only=True`, which keeps the execution machinery and only restricts it with an allow-list whose completeness we must trust. But the format only guarantees "these bytes are tensors". It says nothing about *what those tensors compute*, *where the file came from*, or *what else runs around it*. Things a `.safetensors` checkpoint still cannot protect against:

1. **Adversarial or backdoored weights.** A model can be trained or edited to behave normally except on a trigger input (e.g. misclassify, leak data, emit malicious code suggestions). The weights are still valid float tensors, so the format has no opinion. Detecting this needs evaluation, red-teaming or model auditing, not a file format.
2. **Code that runs alongside the weights.** Hugging Face repos ship `modeling_*.py`, custom tokenizers and configs. `from_pretrained(..., trust_remote_code=True)` executes that Python regardless of whether the weights are safetensors. A malicious repo can simply put its payload in the modelling code instead of in the checkpoint.
3. **Provenance and tampering.** "This file cannot execute code" is a different claim from "this is the file the publisher released". A swapped or modified `.safetensors` (e.g. through a typosquatted repo, a compromised mirror, or a man-in-the-middle download) loads just as safely and just as wrongly. That is solved by hashes, signatures and pinning revisions, which live outside the format.
4. **Resource-exhaustion and parser bugs.** A hostile header can declare huge shapes or overlapping offsets, and the reader library itself could have bugs. The format specifies validation (non-overlapping offsets, size checks), but whether the loader enforces it is an implementation issue, not something the bytes can promise.

Also, safetensors holds weights only. Optimizer state and training metadata still go through some other format (often pickle), so the old risk returns for training checkpoints.


---
## Part 4 — Read a real file

📝 If you set `GGUF_PATH`, run the parse below. Report the architecture and vocab size it
recovered, and which metadata value types your parser could not handle.

**File parsed:** the Ollama blob for `qwen2.5:7b` (4683.1 MB): GGUF v3, 339 tensors, 34 metadata keys, `general.name = Qwen2.5 7B Instruct`, `general.file_type = 15` (15 = Q4_K_M, so quantized tensor types are first-class in the file).

**Architecture recovered from the file itself** (`general.architecture = qwen2`):

| key | value |
|---|---|
| `qwen2.block_count` | 28 |
| `qwen2.context_length` | 32768 |
| `qwen2.embedding_length` | 3584 |
| `qwen2.feed_forward_length` | 18944 |
| `qwen2.attention.head_count` | 28 |
| `qwen2.attention.head_count_kv` | 4 |
| `qwen2.rope.freq_base` | 1000000.0 |
| `qwen2.attention.layer_norm_rms_epsilon` | 9.999999974752427e-07 |

**Tokenizer:** `tokenizer.ggml.model = gpt2` (byte-level BPE), **vocabulary size 152,064** (the length of `tokenizer.ggml.tokens`). The provided parser recovered the same arch and vocab size (`qwen2`, 152,064). This is exactly what makes GGUF different from safetensors: a C++ runtime with no Python can build the graph (28 blocks, GQA with 28 query heads / 4 KV heads, RoPE base 1e6, 32K context) and tokenize text from this one file.

**What the provided parser could not handle.** It decodes only 4 of the 13 GGUF value types (`UINT32`, `FLOAT32`, `STRING`, `ARRAY`) and stops at the first key using anything else. On this file it stopped at `tokenizer.ggml.token_type` after reading 26 of 34 keys, with unhandled tag `[5]` = **INT32**. Everything after that key was never read, including `tokenizer.ggml.add_bos_token` (**BOOL**), the chat template and `general.quantization_version`. The value types present in this file are ARRAY, BOOL, FLOAT32, INT32, STRING, UINT32. What the unhandled types are for:

- **INT32** (seen here): signed integers. `token_type` is an array with one INT32 per vocabulary entry marking each token as normal, unknown, control/special, user-defined, unused or byte. The tokenizer needs it to handle special tokens such as `<|im_start|>` correctly.
- **BOOL** (seen here): flags such as `add_bos_token` / `add_eos_token`, whether to prepend or append special tokens when encoding.
- **UINT64 / INT64**: counts and sizes that can exceed 32 bits (e.g. very large element counts, some token ids or lengths in other models).
- **FLOAT64**: high-precision scalars such as some RoPE scaling or normalization constants.
- **UINT8 / INT8 / UINT16 / INT16**: compact small integers and enum-like fields used by some architectures' metadata.

The extended parser in the cell above implements all 13 types and read all 34 keys of the same file. For a real reader, stopping at an unknown type is a correctness problem, not just a missing feature: keys are length-prefixed but values are not, so a parser that cannot size a value cannot even skip it, and every later key is lost.


In [8]:
def parse_gguf(path):
    UINT32, FLOAT32, STRING, ARRAY = 4, 6, 8, 9
    f = open(path, "rb")
    assert f.read(4) == b"GGUF", "not a GGUF file"
    version, = struct.unpack("<I", f.read(4))
    n_tensors, n_kv = struct.unpack("<QQ", f.read(16))
    def r_str(): return f.read(struct.unpack("<Q", f.read(8))[0]).decode(errors="replace")
    unhandled = set(); meta = {}
    def r_val(tag):
        if tag == UINT32:  return struct.unpack("<I", f.read(4))[0]
        if tag == FLOAT32: return struct.unpack("<f", f.read(4))[0]
        if tag == STRING:  return r_str()
        if tag == ARRAY:
            et, n = struct.unpack("<IQ", f.read(12))
            return [r_val(et) for _ in range(n)]
        unhandled.add(tag); raise ValueError(tag)
    for _ in range(n_kv):
        k = r_str(); tag, = struct.unpack("<I", f.read(4))
        try: meta[k] = r_val(tag)
        except ValueError: break   # hit a type we do not decode; stop cleanly
    f.close()
    arch = meta.get("general.architecture")
    toks = next((v for k, v in meta.items() if k.endswith("tokens")), [])
    return dict(arch=arch, vocab_size=len(toks) if isinstance(toks, list) else None,
                n_kv_read=len(meta), unhandled_type_tags=sorted(unhandled))

if GGUF_PATH and Path(GGUF_PATH).exists():
    RESULTS["gguf_real"] = parse_gguf(GGUF_PATH)
    print(RESULTS["gguf_real"])
else:
    print("GGUF_PATH not set - Part 4 parse skipped (still answer the 📝 cell if you inspected a file elsewhere)")


{'arch': 'qwen2', 'vocab_size': 152064, 'n_kv_read': 26, 'unhandled_type_tags': [5]}


In [9]:
# Part 4, extended: a parser that handles all 13 GGUF metadata value types, run on the same file.
# This shows exactly what the lab's 4-type parser above could not see.
GGUF_TYPES = {0:("UINT8","<B",1), 1:("INT8","<b",1), 2:("UINT16","<H",2), 3:("INT16","<h",2),
              4:("UINT32","<I",4), 5:("INT32","<i",4), 6:("FLOAT32","<f",4), 7:("BOOL","<?",1),
              8:("STRING",None,None), 9:("ARRAY",None,None), 10:("UINT64","<Q",8),
              11:("INT64","<q",8), 12:("FLOAT64","<d",8)}

def parse_gguf_full(path):
    f = open(path, "rb")
    assert f.read(4) == b"GGUF"
    version, = struct.unpack("<I", f.read(4)); n_tensors, n_kv = struct.unpack("<QQ", f.read(16))
    def r_str(): return f.read(struct.unpack("<Q", f.read(8))[0]).decode(errors="replace")
    def r_val(tag, seen):
        seen.add(tag); name, fmt, size = GGUF_TYPES[tag]
        if tag == 8: return r_str()
        if tag == 9:
            et, n = struct.unpack("<IQ", f.read(12))
            return [r_val(et, seen) for _ in range(n)]
        return struct.unpack(fmt, f.read(size))[0]
    meta, key_types = {}, {}
    for _ in range(n_kv):
        k = r_str(); tag, = struct.unpack("<I", f.read(4)); seen = set()
        meta[k] = r_val(tag, seen); key_types[k] = sorted(GGUF_TYPES[t][0] for t in seen)
    f.close()
    return version, n_tensors, meta, key_types

if GGUF_PATH and Path(GGUF_PATH).exists():
    version, n_tensors, meta, key_types = parse_gguf_full(GGUF_PATH)
    arch = meta["general.architecture"]
    hparams = {k.split(".", 1)[1]: v for k, v in meta.items() if k.startswith(arch + ".")}
    lab_types = {"UINT32", "FLOAT32", "STRING", "ARRAY"}
    blind = {k: t for k, t in key_types.items() if set(t) - lab_types}
    stopped_at = next(iter(blind), None)
    RESULTS["gguf_real"].update(dict(
        full_parse=dict(gguf_version=version, n_tensors=n_tensors, n_kv=len(meta),
                        arch=arch, name=meta.get("general.name"), file_type=meta.get("general.file_type"),
                        hparams=hparams, vocab_size=len(meta.get("tokenizer.ggml.tokens", [])),
                        tokenizer_model=meta.get("tokenizer.ggml.model"),
                        value_types_present=sorted({t for ts in key_types.values() for t in ts}),
                        keys_needing_unhandled_types={k: t for k, t in blind.items()},
                        lab_parser_stopped_at=stopped_at)))
    fp = RESULTS["gguf_real"]["full_parse"]
    print(f"GGUF v{version}: {n_tensors} tensors, {len(meta)} metadata keys | {fp['name']} ({arch}), file_type={fp['file_type']}")
    print("hyperparameters:"); [print(f"   {arch}.{k:<34} {v}") for k, v in hparams.items()]
    print("vocab size:", fp["vocab_size"], "| tokenizer:", fp["tokenizer_model"])
    print("value types present in this file:", fp["value_types_present"])
    print("keys the 4-type lab parser cannot decode:"); [print(f"   {k:<40} {t}") for k, t in blind.items()]
    print("lab parser stopped at key:", stopped_at, f"-> read {RESULTS['gguf_real'].get('n_kv_read')} of {len(meta)} keys")
else:
    print("GGUF_PATH not set - extended parse skipped")


GGUF v3: 339 tensors, 34 metadata keys | Qwen2.5 7B Instruct (qwen2), file_type=15
hyperparameters:
   qwen2.block_count                        28
   qwen2.context_length                     32768
   qwen2.embedding_length                   3584
   qwen2.feed_forward_length                18944
   qwen2.attention.head_count               28
   qwen2.attention.head_count_kv            4
   qwen2.rope.freq_base                     1000000.0
   qwen2.attention.layer_norm_rms_epsilon   9.999999974752427e-07
vocab size: 152064 | tokenizer: gpt2
value types present in this file: ['ARRAY', 'BOOL', 'FLOAT32', 'INT32', 'STRING', 'UINT32']
keys the 4-type lab parser cannot decode:
   tokenizer.ggml.token_type                ['ARRAY', 'INT32']
   tokenizer.ggml.add_bos_token             ['BOOL']
lab parser stopped at key: tokenizer.ggml.token_type -> read 26 of 34 keys


---
## Short answers

Fill these from **your own** recorded numbers above — the grader checks they are consistent
with `RESULTS`.

In [10]:
pt, st = RESULTS["formats"]["pt"], RESULTS["formats"]["safetensors"]
sizes, times = RESULTS["stride"]["contig_sizes_mb"], RESULTS["stride"]["contig_times_ms"]
ANSWERS = dict(
    # Part 1  (read off the recorded numbers above, not assumed)
    warm_faster_than_cold = bool(pt["warm_s"] < pt["cold_s"]),                      # for .pt
    safetensors_rss_near_zero = bool(st["load_rss_delta_mb"] < 0.25 * st["file_mb"]),
    # Part 2
    contiguous_cost_grows = bool(times[-1] > times[0] and all(b >= 0.9 * a for a, b in zip(times, times[1:]))),
    # Part 3
    weights_only_blocked_it = bool(RESULTS["trust"]["blocked_by_weights_only"] and not RESULTS["trust"]["payload_fired_default"]),
    safetensors_still_cannot = ("backdoored/adversarial weights (format only guarantees bytes are tensors, not what they compute); "
                                "trust_remote_code modelling code that runs regardless of weight format; "
                                "provenance/tampering (needs hashes/signatures, not a format)"),
)
print(ANSWERS)


{'warm_faster_than_cold': True, 'safetensors_rss_near_zero': True, 'contiguous_cost_grows': True, 'weights_only_blocked_it': True, 'safetensors_still_cannot': 'backdoored/adversarial weights (format only guarantees bytes are tensors, not what they compute); trust_remote_code modelling code that runs regardless of weight format; provenance/tampering (needs hashes/signatures, not a format)'}


---
## Export — run last

In [11]:
env = dict(platform=platform.platform(), python=sys.version.split()[0],
           torch=torch.__version__, safetensors=safetensors.__version__, linux=IS_LINUX)
sub = dict(roll=ROLL_NUMBER, name=NAME, env=env, results=RESULTS, answers=ANSWERS)

out = Path(f"submission_lab11_12_{ROLL_NUMBER}.json")
out.write_text(json.dumps(sub, indent=2))

# report what was recorded
fmts = list(RESULTS["formats"])
rec = [f"formats={fmts}", f"stride keys={list(RESULTS['stride'])}",
       f"trust keys={list(RESULTS['trust'])}", f"gguf_real={'yes' if RESULTS['gguf_real'] else 'skipped'}"]
missing = [k for k, val in ANSWERS.items() if val in (None, "")]
print("wrote", out, "\n  " + "\n  ".join(rec))
print("  UNFILLED ANSWERS:", missing or "none")
assert "pt" in fmts and "safetensors" in fmts, "Part 1 must record at least .pt and .safetensors"


wrote submission_lab11_12_202518030.json 
  formats=['pt', 'safetensors', 'gguf']
  stride keys=['transpose_same_ptr', 'contig_sizes_mb', 'contig_times_ms', 'safetensors_refused_view']
  trust keys=['payload_fired_default', 'blocked_by_weights_only', 'payload_fired_unsafe']
  gguf_real=yes
  UNFILLED ANSWERS: none
